In [1]:
import os
os.chdir(os.path.join(os.getcwd(), '..'))
os.getcwd()

'd:\\learning\\learning\\programing\\portfolio_projects\\1_image'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


In [3]:
test_df = pd.read_csv('data/preprocessed_data/final_data/test.csv')
final_df = pd.read_csv('data/preprocessed_data/final_dataframe.csv')

In [ ]:
full_test_df = test_df.merge(final_df, on = ['empty_slice', 'prev_path', 'path', 'next_path'])
full_test_df = full_test_df.drop(['large_bowel', 'small_bowel', 'stomach', 'height', 'width'], axis = 1)

In [6]:
full_test_df.head()

,empty_slice,path,prev_path,next_path,mask_path,collapsed_id,case,slice
0,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,19,16,1
1,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,19,16,2
2,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,19,16,3
3,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,19,16,4
4,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,19,16,5


In [7]:
full_test_df = full_test_df[['collapsed_id', 'case', 'slice', 'empty_slice', 'path', 'prev_path', 'next_path', 'mask_path']]

In [8]:
d = Path(full_test_df.mask_path[0])

In [9]:
import re


new_stem = re.sub(r"_\d+$", "", d.stem)
new_path = d.with_name(new_stem + '.pt')

In [10]:
Path(new_path.parent.with_name('collapsed_mask'), new_path.name)

WindowsPath('data/preprocessed_data/final_data/collapsed_mask/19_16.pt')

In [11]:
full_test_df['mask_path'] = full_test_df['mask_path'].apply(lambda p: Path(p))

In [ ]:
def make_collapsed_mask_path(d):
    p = d.with_name(re.sub(r'_\d+$', '', d.stem) + '.pt')
    new_path = p.parent.with_name('post_model')/ 'collapsed_mask' / p.name
    return new_path
full_test_df['collapsed_mask_path'] = full_test_df['mask_path'].apply(make_collapsed_mask_path)

In [13]:
full_test_df['collapsed_mask_path'] = full_test_df['mask_path'].apply(make_collapsed_mask_path)

In [14]:
full_test_df['collapsed_mask_path'][0]

WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/19_16.pt')

In [15]:
def make_model_mask_path(d):
    p = d.with_name( d.stem + '.pt')
    new_path = p.parent.with_name('post_model')/ 'model_mask' / p.name
    return new_path


full_test_df['model_mask_path'] = full_test_df['mask_path'].apply(make_model_mask_path)
full_test_df['model_mask_path'][0]

WindowsPath('data/preprocessed_data/final_data/post_model/model_mask/19_16_1.pt')

In [16]:
def make_out_transformed_mask_path(d):
    p = d.with_name(d.stem + '.pt')
    new_path = p.parent.with_name('post_model')/ 'out_transformed_mask' / p.name
    return new_path


full_test_df['out_transformed_mask_path'] = full_test_df['mask_path'].apply(make_out_transformed_mask_path)
full_test_df['out_transformed_mask_path'][0]

WindowsPath('data/preprocessed_data/final_data/post_model/out_transformed_mask/19_16_1.pt')

In [17]:
a = np.load(full_test_df.mask_path[0])

In [18]:
b = []
b.append(a)
b.append(a)

In [19]:
np.shape(np.sum(b, axis = 0))

(3, 310, 360)

In [20]:
import torch
from src.components.optuna_model_selection.model import MRIFlexAttentionUNet
from src.components.optuna_model_selection.dataset import MRISaverDataset
from torchvision.transforms import v2
from torch.utils.data import DataLoader

In [ ]:


class ThresholdCalibrator:
    def __init__(self, config):
self.threshold = torch.Tensor([0.9,0.9,0.9])
        self.device =torch.device('cuda' if config.accelerator == 'gpu' else 'cpu') 
        state_dict = torch.load(config.inp_model_path, 
                                self.device, weights_only=False)['state_dict']
        inner_state_dict = {
            k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")
        }

        self.model = MRIFlexAttentionUNet( config.inp_channels,
            config.first_conv_out_channels,
            config.num_classes,
            config.depth,
            config.n_encoder_conv_layers,
            config.n_decoder_conv_layers,
            config.kernel_sizes)
        
        
        self.model.load_state_dict(inner_state_dict)

        dataset = MRISaverDataset(config.full_csv,
                            transform=v2.Compose([
                                        v2.Resize(256),
                                        v2.CenterCrop(256)
                                    ])
                                    )
        
        self.dataloader = DataLoader(dataset, config.batch  , False, num_workers= config.num_workers)
        prev_collapsed_path = dataset[0]

        with torch.no_grad():
            for i, batch in enumerate(self.dataloader):
                inp, idx = batch
                out_mask_path = dataset.partial_df.model_mask_path[[idx]]
                inp = inp.to(self.device)
              
                out = self.model(inp)
                for j in range(len(out)):
                    if not out_mask_path[j].parent.exists():
                        out_mask_path[j].parent.mkdir(parents=True, exist_ok = True)
                    if not out_mask_path[j].exists():
                        torch.save(out[j], out_mask_path[j])

            df = config.full_csv.copy()
            
            df = df[~df.empty_slice]
            grooped_obj = [p for p in df.groupby('collapsed_mask_path')['model_mask_path']]

            for gr in grooped_obj:
                save = gr[0]
                collapsed_mask = []
                model_mask_path  = gr[1].to_numpy()

                for j in idx:
                    p = full_test_df.model_mask_path[int(j)]
                    out = torch.load(p, 'cpu')
                    probs = torch.sigmoid(out) 
                    thresh = self.threshold.view(1, -1, 1, 1)
                    mask = (probs >= thresh).int()
                    collapsed_mask.append(mask)

                collapsed_mask = torch.sum(collapsed_mask, 0)
                if not save.parent.exists():
                    save.parent.mkdir(parents=True, exist_ok = True)
                if not save.exists():
                    torch.save(collapsed_mask, save)

                


In [22]:
dataset = MRISaverDataset(full_test_df,
                            transform=v2.Compose([
                                        v2.Resize(256),
                                        v2.CenterCrop(256)
                                    ]))

In [23]:
dataset[0][1]

0

In [24]:
df = full_test_df.copy()

In [25]:
for i, col_p in enumerate(df.collapsed_id.unique)
    [torch.load(p[i]) for p, e in df.groupby('collapsed_mask_path')[['path', 'empti_slice']] if e[i]]

SyntaxError: expected ':' (270204435.py, line 1)

In [ ]:
[(p, e) for p, e in df.groupby('collapsed_mask_path')[['path', 'empty_slice']] ][0][1]

,path,empty_slice
221,data\extracted_data\train\case78\case78_day0\s...,False
222,data\extracted_data\train\case78\case78_day0\s...,False
223,data\extracted_data\train\case78\case78_day0\s...,False
224,data\extracted_data\train\case78\case78_day0\s...,False
225,data\extracted_data\train\case78\case78_day0\s...,False
226,data\extracted_data\train\case78\case78_day0\s...,False
227,data\extracted_data\train\case78\case78_day0\s...,False
228,data\extracted_data\train\case78\case78_day0\s...,False
229,data\extracted_data\train\case78\case78_day0\s...,False
230,data\extracted_data\train\case78\case78_day0\s...,False


In [ ]:


class SSS:
    def __init__(self, config):

        self.device =torch.device('cuda' if config.accelerator == 'gpu' else 'cpu')

        ckpt = torch.load(config.base_model_path, 
                                self.device, weights_only=False)
        state_dict = ckpt['state_dict']
        inner_state_dict = {
            k[len("model."):]: v for k, v in state_dict.items() if k.startswith("model.")
        }

        hparams =ckpt['hyper_parameters']
        first_conv_out_channels = hparams['first_conv_out_channels']
        depth = hparams['depth']
        n_encoder_conv_layers = hparams['n_encoder_conv_layers']
        n_decoder_conv_layers = hparams['n_decoder_conv_layers']
        kernel_sizes = hparams['kernel_sizes']
        self.threshold = torch.from_numpy(np.load(config.threshold_path))

        self.model = MRIFlexAttentionUNet(config.inp_channels,
            first_conv_out_channels,
            config.num_classes,
            depth,
            n_encoder_conv_layers,
            n_decoder_conv_layers,
            kernel_sizes)
        
        self.model.load_state_dict(inner_state_dict)

        self.make_post_model_dfs(config.train_cdv, config.dev_csv, config.test_csv)
        self.dataset = MRISaverDataset(self.df,transform=config.transform)
        self.dataloader = DataLoader(dataset, config.batch_size , False, num_workers= config.num_workers)



    def make_post_model_dfs(self, train_df_path, val_df_path, test_df_path, detailed_df_path):
        
        train_df = self._make_df(train_df_path, detailed_df_path)
        val_df = self._make_df(val_df_path, detailed_df_path)
        test_df = self._make_df(test_df_path, detailed_df_path)
        self.df = pd.concat([train_df, val_df, test_df], ignore_index= True)



    def _make_df(self, df_path: Path, detailed_df_path):

        full_df_path = df_path.parent / (df_path.stem + '_post_model') / df_path.suffix

        if full_df_path.exists:
            full_df = pd.read_csv(full_df_path)


        else:
            df = pd.read_csv(df_path)
            detailed_df = pd.read_csv(detailed_df_path)

            full_df = df.merge(detailed_df, on = ['empty_slice', 'prev_path', 'path', 'next_path'])
            full_df = full_df.drop(['large_bowel', 'small_bowel', 'stomach', 'height', 'width'], axis = 1)
            full_df = full_df[['collapsed_id', 'case', 'slice', 'empty_slice', 'path', 'prev_path', 'next_path', 'mask_path']]
            full_df['mask_path'] = full_df['mask_path'].apply(lambda p: Path(p))


            def make_collapsed_mask_path(d):
                p = d.with_name(re.sub(r'_\d+$', '', d.stem) + '.pt')
                new_path = p.parent.with_name('post_model')/ 'collapsed_mask' / p.name
                return new_path
            
            full_df['collapsed_mask_path'] = full_df['mask_path'].apply(make_collapsed_mask_path)
            del make_model_mask_path


            def make_model_mask_path(d):
                p = d.with_name( d.stem + '.pt')
                new_path = p.parent.with_name('post_model')/ 'model_mask' / p.name
                return new_path

            full_df['model_mask_path'] = full_df['mask_path'].apply(make_model_mask_path)
            del make_model_mask_path


            def make_out_transformed_mask_path(d):
                p = d.with_name(d.stem + '.pt')
                new_path = p.parent.with_name('post_model')/ 'out_transformed_mask' / p.name
                return new_path

            full_df['out_transformed_mask_path'] = full_df['mask_path'].apply(make_out_transformed_mask_path)
            del make_out_transformed_mask_path

            full_df.to_csv(full_df_path, index = False)

        return full_df


    def save_model_and_out_mask(self):
        
        self.model.to(self.device)
        self.model.eval()
        with torch.no_grad():
            for batch in self.dataloader:
                inp, idx = batch
                
                inp = inp.to(self.device)
                out = self.model(inp)

                for j in range(len(out)):
                    out_mask_path = dataset.partial_df.model_mask_path[int(idx[j])]
                    if not out_mask_path.parent.exists():
                        out_mask_path.parent.mkdir(parents=True, exist_ok = True)
                    if not out_mask_path.exists():
                        torch.save(out[j], out_mask_path)

               

    def save_collapsed_mask(self):
        
        df = self.dataset.partial_df[~self.dataset.partial_df.empty_slice]    
        grooped_obj = [p for p in df.groupby('collapsed_mask_path', sort = False)['model_mask_path']]
    
        for gr in grooped_obj:
            save = gr[0]
            
            collapsed_mask = []
            model_mask_path  = gr[1].to_numpy()
            
            k = 0 
            for p in model_mask_path:
                out = torch.load(p, self.device)
                probs = torch.sigmoid(out) 
                thresh = self.threshold.view(1, -1, 1, 1)
                mask = (probs >= thresh).int()
                collapsed_mask.append(mask)

            collapsed_mask = np.array(collapsed_mask)
            collapsed_mask = torch.from_numpy(collapsed_mask)
            collapsed_mask = torch.sum(collapsed_mask, dim = 0)
            print(collapsed_mask.shape)
            if not save.parent.exists():
                save.parent.mkdir(parents=True, exist_ok = True)
            if not save.exists():
                torch.save(collapsed_mask, save)
            break

                


In [114]:
SSS()

torch.Size([1, 3, 256, 256])


In [43]:
full_test_df.iloc[[0,1,2,3]]

,collapsed_id,case,slice,empty_slice,path,prev_path,next_path,mask_path,collapsed_mask_path,model_mask_path,out_transformed_mask_path
0,19,16,1,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,data\preprocessed_data\final_data\post_model\c...,data\preprocessed_data\final_data\post_model\m...,data\preprocessed_data\final_data\post_model\o...
1,19,16,2,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,data\preprocessed_data\final_data\post_model\c...,data\preprocessed_data\final_data\post_model\m...,data\preprocessed_data\final_data\post_model\o...
2,19,16,3,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,data\preprocessed_data\final_data\post_model\c...,data\preprocessed_data\final_data\post_model\m...,data\preprocessed_data\final_data\post_model\o...
3,19,16,4,True,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...,data\preprocessed_data\final_data\post_model\c...,data\preprocessed_data\final_data\post_model\m...,data\preprocessed_data\final_data\post_model\o...


In [79]:
df = full_test_df[~full_test_df.empty_slice]
            
            
grooped_obj = [p for p in df.groupby('collapsed_mask_path', sort = False)['model_mask_path']]

In [80]:
grooped_obj[0][1].to_numpy()[0]

WindowsPath('data/preprocessed_data/final_data/post_model/model_mask/19_16_26.pt')

In [74]:
grooped_obj[0][0]

WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/118_78.pt')

In [119]:
full_test_df.model_mask_path.iloc[0].parent / (full_test_df.model_mask_path.iloc[0].stem + '_post_model') / full_test_df.model_mask_path.iloc[0].suffix

WindowsPath('data/preprocessed_data/final_data/post_model/model_mask/19_16_1_post_model/.pt')

In [127]:
df.collapsed_mask_path.unique()

array([WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/19_16.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/118_78.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/119_78.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/120_78.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/121_78.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/122_78.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/188_121.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/189_121.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/190_121.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/collapsed_mask/236_140.pt'),
       WindowsPath('data/preprocessed_data/final_data/post_model/co